Helper cell

In [ ]:
import heapq
import json
import math
import os
import random

# Directories inside workspace
BASE_DIR = os.path.expanduser('~/amr_ws/src/warehouse_sim')
MAPS_DIR = os.path.join(BASE_DIR, 'maps')
WORLDS_DIR = os.path.join(BASE_DIR, 'worlds')
os.makedirs(MAPS_DIR, exist_ok=True)
os.makedirs(WORLDS_DIR, exist_ok=True)

# Arena specifications (12m x 12m at 0.05m/px = 240x240 occupancy grid)
MAP_W = 240
MAP_H = 240
RES = 0.05
ORIGIN_X = -6.0
ORIGIN_Y = -6.0
ROBOT_RADIUS = 0.18  # Safety buffer

def world_to_grid(x_m, y_m):
    """Converts meters -> grid pixels"""
    gx = int((x_m - ORIGIN_X) / RES)
    gy = int((y_m - ORIGIN_Y) / RES)
    return gx, gy

def make_empty_arena_grid():
    """Initializes 240x240 blank grid (free space = 255), solid outer walls (0)."""
    grid = [[255 for _ in range(MAP_W)] for _ in range(MAP_H)]
    for x in range(MAP_W):
        grid[0][x] = 0
        grid[MAP_H - 1][x] = 0
    for y in range(MAP_H):
        grid[y][0] = 0
        grid[y][MAP_W - 1] = 0
    return grid

def write_map_files(prefix, grid):
    """
    Saves 2D grid as a ROS 2 compatible .pgm image, .json data is saved as a .yaml file
    """
    # Flips Y-Axis when writing PGM so orientation of top and bottom matches ROS map
    pgm_data = bytearray(f'P5\n{MAP_W} {MAP_H}\n255\n'.encode())
    for y in range(MAP_H - 1, -1, -1):
        pgm_data.extend(grid[y])
    
    with open(os.path.join(MAPS_DIR, f'{prefix}.pgm'), 'wb') as f:
        f.write(pgm_data)
        
    with open(os.path.join(MAPS_DIR, f'{prefix}.yaml'), 'w') as f:
        f.write(
            f'image: {prefix}.pgm\n'
            f'resolution: {RES}\n'
            f'origin: [{ORIGIN_X}, {ORIGIN_Y}, 0.0]\n'
            f'negate: 0\n'
            f'occupied_thresh: 0.65\n'
            f'free_thresh: 0.19\n'
        )

def get_outer_walls_sdf():
    """Generates the 4 bounding walls"""
    walls = [
        ('wall_n', '0 6.05 0.5 0 0 0', '12.2 0.1 1.0'),
        ('wall_s', '0 -6.05 0.5 0 0 0', '12.2 0.1 1.0'),
        ('wall_e', '6.05 0 0.5 0 0 0', '0.1 12.2 1.0'),
        ('wall_w', '-6.05 0 0.5 0 0 0', '0.1 12.2 1.0'),
    ]
    out = []
    for name, pose, size in walls:
        out.append(f'''    <model name="{name}">
      <static>true</static>
      <pose>{pose}</pose>
      <link name="link">
        <collision name="collision"><geometry><box><size>{size}</size></box></geometry></collision>
        <visual name="visual">
          <geometry><box><size>{size}</size></box></geometry>
          <material><ambient>0.15 0.15 0.15 1</ambient><diffuse>0.3 0.3 0.3 1</diffuse></material>
        </visual>
      </link>
    </model>''')
    return '\n'.join(out)

print(f"Workspace environment loaded successfully at: {BASE_DIR}")

Workspace environment loaded successfully at: C:\Users\Nyaan/amr_ws/src/warehouse_sim


Level 1 - Map Generator
generate 30 batched Level 1 Trials with randomized Position & Rotation

In [ ]:
import random
import math
import json
import os

def generate_level1_batch(start_seed=0, count=30):
    for s in range(start_seed, start_seed + count):
        rng = random.Random(s)
        grid = make_empty_arena_grid()

        # 1. Jitter obstacle center position and rotation (yaw)
        cx = rng.uniform(-1.5, 1.5)
        cy = rng.uniform(-1.5, 1.5)
        obs_w = rng.uniform(1.6, 2.2)
        obs_d = rng.uniform(1.6, 2.2)
        yaw = rng.uniform(0.0, math.pi)  # random rotation

        # 2. Rasterize rotated box onto the occupancy grid
        cos_y = math.cos(-yaw)
        sin_y = math.sin(-yaw)
        hw, hd = obs_w / 2.0, obs_d / 2.0
        bound = 0.5 * math.hypot(obs_w, obs_d)  # bounding radius for pixel search

        x_min, y_min = world_to_grid(cx - bound, cy - bound)
        x_max, y_max = world_to_grid(cx + bound, cy + bound)

        for y in range(max(0, y_min), min(MAP_H, y_max + 1)):
            for x in range(max(0, x_min), min(MAP_W, x_max + 1)):
                wx = ORIGIN_X + (x + 0.5) * RES - cx
                wy = ORIGIN_Y + (y + 0.5) * RES - cy
                lx = wx * cos_y - wy * sin_y 
                ly = wx * sin_y + wy * cos_y
                if abs(lx) <= hw and abs(ly) <= hd:
                    grid[y][x] = 0

        # File naming matching
        trial_id = f"s{s:02d}"
        prefix = f"level1_{trial_id}"
        write_map_files(prefix, grid)

        # 3. Generate a start & goal across the obstacle
        angle_line = rng.uniform(0, 2 * math.pi)
        start_m = (cx - 4.5 * math.cos(angle_line), cy - 4.5 * math.sin(angle_line))
        goal_m = (cx + 4.5 * math.cos(angle_line), cy + 4.5 * math.sin(angle_line))
        start_yaw = math.atan2(goal_m[1] - start_m[1], goal_m[0] - start_m[0])

        # 4. Create Gazebo SDF World with the obstacle
        sdf_text = f'''<?xml version="1.0" ?>
<sdf version="1.8">
  <world name="{prefix}">
    <physics name="1ms" type="ignored"><max_step_size>0.001</max_step_size><real_time_factor>1.0</real_time_factor></physics>
    <plugin filename="gz-sim-physics-system" name="gz::sim::systems::Physics"/>
    <plugin filename="gz-sim-sensors-system" name="gz::sim::systems::Sensors"><render_engine>ogre2</render_engine></plugin>
    <plugin filename="gz-sim-scene-broadcaster-system" name="gz::sim::systems::SceneBroadcaster"/>
    <light name="sun" type="directional"><cast_shadows>true</cast_shadows><pose>0 0 10 0 0 0</pose><diffuse>0.8 0.8 0.8 1</diffuse><specular>0.2 0.2 0.2 1</specular><direction>-0.5 0.1 -0.9</direction></light>
    <model name="ground_plane">
      <static>true</static>
      <link name="link">
        <collision name="collision"><geometry><plane><normal>0 0 1</normal><size>12 12</size></plane></geometry></collision>
        <visual name="visual"><geometry><plane><normal>0 0 1</normal><size>12 12</size></plane></geometry><material><ambient>0.85 0.85 0.85 1</ambient><diffuse>0.85 0.85 0.85 1</diffuse></material></visual>
      </link>
    </model>
{get_outer_walls_sdf()}
    <model name="central_obstacle">
      <static>true</static>
      <pose>{cx:.3f} {cy:.3f} 0.5 0 0 {yaw:.3f}</pose>
      <link name="link">
        <collision name="collision"><geometry><box><size>{obs_w:.3f} {obs_d:.3f} 1.0</size></box></geometry></collision>
        <visual name="visual">
          <geometry><box><size>{obs_w:.3f} {obs_d:.3f} 1.0</size></box></geometry>
          <material><ambient>0.25 0.25 0.3 1</ambient><diffuse>0.45 0.45 0.5 1</diffuse></material>
        </visual>
      </link>
    </model>
  </world>
</sdf>'''
        with open(os.path.join(WORLDS_DIR, f'{prefix}.sdf'), 'w') as f:
            f.write(sdf_text)

        # save coordinate as .json
        meta = {
             'trial': trial_id, 'seed': s,
             'obstacle': {'x': round(cx, 3), 'y': round(cy, 3), 'yaw': round(yaw, 3), 'w': round(obs_w, 3), 'd': round(obs_d, 3)},
             'start': {'x': round(start_m[0], 3), 'y': round(start_m[1], 3), 'yaw': round(start_yaw, 3)},
             'goal': {'x': round(goal_m[0], 3), 'y': round(goal_m[1], 3)}
        }
        with open(os.path.join(MAPS_DIR, f'{prefix}_trial.json'), 'w') as f:
            json.dump(meta, f, indent=2)

        print(f"Generated Level 1 [{trial_id}] (Seed {s}) | Obstacle Yaw: {math.degrees(yaw):.1f}°")

# Run all trials specified 
generate_level1_batch(start_seed=0, count=30)

Generated Level 1 [s00] (Seed 0) | Obstacle Yaw: 92.0°
Generated Level 1 [s01] (Seed 1) | Obstacle Yaw: 89.2°
Generated Level 1 [s02] (Seed 2) | Obstacle Yaw: 150.4°
Generated Level 1 [s03] (Seed 3) | Obstacle Yaw: 112.6°
Generated Level 1 [s04] (Seed 4) | Obstacle Yaw: 12.0°
Generated Level 1 [s05] (Seed 5) | Obstacle Yaw: 133.2°
Generated Level 1 [s06] (Seed 6) | Obstacle Yaw: 0.1°
Generated Level 1 [s07] (Seed 7) | Obstacle Yaw: 96.5°
Generated Level 1 [s08] (Seed 8) | Obstacle Yaw: 15.3°
Generated Level 1 [s09] (Seed 9) | Obstacle Yaw: 1.2°
Generated Level 1 [s10] (Seed 10) | Obstacle Yaw: 146.4°
Generated Level 1 [s11] (Seed 11) | Obstacle Yaw: 91.4°
Generated Level 1 [s12] (Seed 12) | Obstacle Yaw: 2.0°
Generated Level 1 [s13] (Seed 13) | Obstacle Yaw: 33.4°
Generated Level 1 [s14] (Seed 14) | Obstacle Yaw: 48.8°
Generated Level 1 [s15] (Seed 15) | Obstacle Yaw: 177.5°
Generated Level 1 [s16] (Seed 16) | Obstacle Yaw: 73.7°
Generated Level 1 [s17] (Seed 17) | Obstacle Yaw: 137.9°